# Inspecting the circuits of the experiments (Sec. VI)

This notebook shows, for any configuration of the three experiments,
* the **logical circuit** built by `circuits.py` and its check against the logical circuit we executed (`qpy/<experiment>_logical.qpy`),
* the **compiled (ISA) circuit** that ran on `ibm_fez` (`qpy/<experiment>_isa.qpy`), with its qubit layout, depth and gate counts,
* the measured **counts** of that circuit and the resulting accuracy of the adversary (`data/hardware/<experiment>.json`),
* the **simulation circuit** used by `do_simulation.py`.

Choose the configuration in the cell *Choose a configuration* and run all cells below it.

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
from qiskit import qpy
from qiskit_aer.noise import pauli_error

import circuits as C
import plot_figures as P

def load_qpy(experiment, stage):
    with open(f"qpy/{experiment}_{stage}.qpy", "rb") as stream:
        return {(qc.metadata["configuration"], qc.metadata["n"], qc.metadata["secret"]): qc
                for qc in qpy.load(stream)}

LOGICAL = {e: load_qpy(e, "logical") for e in C.EXPERIMENTS}
ISA = {e: load_qpy(e, "isa") for e in C.EXPERIMENTS}
HARDWARE = {e: {(r["configuration"], r["n"]): r for r in P.load("hardware", e)} for e in C.EXPERIMENTS}

## Available configurations

| experiment | configuration | mechanism | adversary | n on hardware |
|---|---|---|---|---|
| `motivating` (Fig. 2) | `XZ_NR` / `XZ_EA` | $\mathcal E_{XZ}$, computation $X^x$ | no reference / entanglement attack | 1 |
| `attack` (Fig. 4) | `XZ_BV` / `XZ_EA` | $\mathcal E_{XZ,q}^{\otimes n}$, total QDP 3 | BV, no reference / entanglement attack | 1–10 |
| `attack` (Fig. 4) | `DP_BV` / `DP_EA` | $\mathcal D_p^{\otimes n}$, total QDP 3 | BV, no reference / entanglement attack | 1–10 |
| `defense` (Fig. 5) | `DP_EA` | $\mathcal D_p^{\otimes n}$, total EA-QDP 3 | entanglement attack | 1–10 |
| `defense` (Fig. 5) | `DP_GLOBAL_EA` | $\mathcal D_p^{\rm glob}$, total EA-QDP 3 | entanglement attack | 1–6 |

The secret strings of each $n$ are listed below (all strings for $n\le3$, 8 sampled strings for $n\ge4$).

In [ ]:
for n in C.N_RANGE:
    print(f"n={n:2d}:", " ".join(C.secret_strings(n)))

## Choose a configuration

In [ ]:
EXPERIMENT = "attack"       # "motivating", "attack" or "defense"
CONFIGURATION = "XZ_EA"     # see the table above
N = 2                       # number of sensitive bits (qubit pairs)
SECRET = C.secrets_for(EXPERIMENT, N)[1]

x = C.mechanism_parameter(EXPERIMENT, CONFIGURATION, N)
eps = C.per_bit_epsilon(EXPERIMENT, CONFIGURATION, N)
framework = "EA-QDP" if EXPERIMENT == "defense" else "QDP"
print(f"{EXPERIMENT} / {CONFIGURATION} / n = {N} / secret x = {SECRET}")
print(f"mechanism parameter {'q' if CONFIGURATION.startswith('XZ') else 'p'} = {x:.6f}")
print(f"total {framework} parameter = {C.total_epsilon(EXPERIMENT, CONFIGURATION, N):.6f}")
print(f"per-bit {framework} parameter = {eps:.6f}  ->  accuracy bound e^eps/(1+e^eps) = {C.accuracy_bound(eps):.4f}")

## Logical circuit

Registers: `r` = reference qubits $R_i$ kept by the adversary, `s` = system qubits $S_i$ submitted to the
quantum computer; `env` (for $\mathcal E_{XZ,q}$) or `a`, `b`, `f` (for $\mathcal D_p$) are the environment qubits that
realize the mechanism. Barriers separate the state preparation, the oracle (computation), the mechanism
(`curate`) and the adversary's measurement (`decode`). The circuit rebuilt by `circuits.py` is compared with the
circuit stored in `qpy/`, i.e. the one that was compiled and executed.

In [ ]:
logical = C.hardware_circuit(EXPERIMENT, CONFIGURATION, SECRET)
print("identical to the executed logical circuit:", logical == LOGICAL[EXPERIMENT][CONFIGURATION, N, SECRET])
logical.draw("mpl", fold=40, scale=.7)

## Compiled (ISA) circuit executed on `ibm_fez`

Compiled with `optimization_level=1`, `seed_transpiler=41` into the native gates of `ibm_fez`
(`rz`, `sx`, `x`, `cz`, `measure`). Only the qubits used by the circuit are drawn.

In [ ]:
isa = ISA[EXPERIMENT][CONFIGURATION, N, SECRET]
layout = isa.layout.initial_index_layout(filter_ancillas=True)
wires = [f"{reg.name}[{i}]" for reg in logical.qregs for i in range(reg.size)]
print("logical qubit -> physical qubit:", dict(zip(wires, layout)))
print("depth:", isa.depth(), "| gate counts:", dict(isa.count_ops()))
isa.draw("mpl", idle_wires=False, fold=40, scale=.6)

## Measured counts and accuracy of this circuit

Count keys are `"<reference bits>|<system bits>"`, where character $i$ of each half belongs to qubit pair $i$
(the same index as character $i$ of the secret string). The decision rule of `plot_figures.py` is applied to
every shot; the accuracy over all secret strings of the configuration is the point plotted in the paper.

In [ ]:
record = HARDWARE[EXPERIMENT].get((CONFIGURATION, N))
if record is None:
    print("not run on hardware")
else:
    run = next(r for r in record["runs"] if r["secret"] == SECRET)
    top = Counter(run["counts"]).most_common(8)
    print(f"{run['shots']} shots, {len(run['counts'])} distinct outcomes; most frequent:")
    for key, count in top:
        answered, guesses = P.decide(EXPERIMENT, CONFIGURATION, key)
        guess = "".join(g if a else "-" for a, g in zip(answered, guesses))
        print(f"  {key:>25s}  {count:5d}   guess {guess}")
    table = P.shot_table(EXPERIMENT, CONFIGURATION, run)
    answered = sum(a*c for (a, _), c in table.items())
    correct = sum(k*c for (_, k), c in table.items())
    print(f"this secret: {correct}/{answered} answered bits correct = {correct/answered:.4f}")
    point = P.accuracy(record, seed=0)
    print(f"configuration (all secrets): accuracy = {point['accuracy']:.4f}, answer rate = {point['answer_rate']:.4f}")

## Simulation circuit

`do_simulation.py` does not simulate the environment qubits. It inserts the Pauli channel of one copy of the
mechanism on each system qubit (drawn as `quantum_channel`) and samples Pauli trajectories with the Aer
statevector simulator; for $\mathcal D_p^{\rm glob}$ one keep/replace decision is shared by all qubits of a shot.
The entanglement attack of the motivating example (`XZ_EA`) is simulated with the logical circuit above.

In [ ]:
if EXPERIMENT == "motivating" and CONFIGURATION == "XZ_EA":
    print("simulated with the logical circuit above")
elif EXPERIMENT == "motivating":
    branches = C.pauli_branches(CONFIGURATION, x)
    print("Pauli branches on the system qubit:", [(b, round(w, 6)) for b, w in branches])
    display(C.motivating_no_reference_simulation_circuit(SECRET, pauli_error(branches).to_instruction()).draw("mpl", scale=.7))
else:
    branches = C.pauli_branches(CONFIGURATION, x) if CONFIGURATION != "DP_GLOBAL_EA" else [(s, .25) for s in "IXYZ"]
    print("Pauli branches per system qubit:", [(b, round(w, 6)) for b, w in branches])
    sim_circuit = C.simulation_circuit(SECRET, CONFIGURATION, pauli_error(branches).to_instruction())
    display(sim_circuit.draw("mpl", scale=.7))

## Check every executed circuit

Rebuild all logical circuits with `circuits.py` and compare them with the executed ones; check that every
compiled circuit uses the initial layout recorded in `data/hardware/`.

In [ ]:
for experiment in C.EXPERIMENTS:
    same = sum(C.hardware_circuit(experiment, c, s) == qc for (c, n, s), qc in LOGICAL[experiment].items())
    layouts = {(r["configuration"], r["n"], u["secret"]): u["initial_layout"]
               for r in HARDWARE[experiment].values() for u in r["runs"]}
    lay = sum(qc.layout.initial_index_layout(filter_ancillas=True) == layouts[k] for k, qc in ISA[experiment].items())
    print(f"{experiment:10s}: {same}/{len(LOGICAL[experiment])} logical circuits identical, "
          f"{lay}/{len(ISA[experiment])} ISA layouts as recorded")